In [1]:

import numpy as np 
import pandas as pd 
from pathlib import Path
import pickle 
import whisper
from tqdm.auto import tqdm
import argparse
from num2words import num2words
from collections.abc import Iterable
import librosa 
import re 

from corpus.binaural_swc_currated_pd import get_window_bounds

/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_whisper/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def flatten(xs):
    for x in xs:
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def convert_transcript(tscrpt):
    tscrpt = tscrpt.lower()
    tscrpt = re.sub(r'[^a-z0-9 ]+', '', tscrpt)
    tscrpt = [num2words(token).split('-') if token.isdigit() else token for token in tscrpt.split(' ')]
    return list(flatten(tscrpt))



In [3]:
parent_dir = Path("/om/user/imgriff/datasets/human_word_rec_SWC_2024/")
manifest = pd.read_pickle(parent_dir / "full_cue_target_distractor_df_w_meta_paths.pdpkl")
out_manifest_name = parent_dir /"full_cue_target_distractor_df_w_meta_transcripts.pdpkl"
# check if the manifest already exists
if out_manifest_name.exists():
    print("Manifest already exists")

# get file names as lists
src_fns = manifest.excerpt_src_fn.to_list()
distractor_1_fns = manifest.excerpt_distractor_1_src_fn.to_list()
distractor_2_fns = manifest.excerpt_distractor_2_src_fn.to_list()


Manifest already exists


In [4]:
manifest.head()

,orig_df_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,...,cue_split_int,cue_sr,cue_src_fn,cue_total_file_duration_in_s,cue_word,sex_cond,excerpt_src_fn,excerpt_cue_src_fn,excerpt_distractor_1_src_fn,excerpt_distractor_2_src_fn
0,601538,laura-s,0.29,1205.07,1204.78,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,1796.176689,each,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
1,638828,dolliellama,0.36,737.92,737.56,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,4871.649524,behavior,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
2,249863,ama1016,0.52,371.41,370.89,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,language,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
3,127418,popularoutcast,0.47,3399.30,3398.83,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,1526.788934,ties,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
4,961747,sedola,0.42,1898.92,1898.50,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,2335.023311,from,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...


In [5]:
print("Placing model on GPU")
model = whisper.load_model("medium").cuda()
options = whisper.DecodingOptions()


Placing model on GPU


In [6]:
manifest.head()

,orig_df_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,...,cue_split_int,cue_sr,cue_src_fn,cue_total_file_duration_in_s,cue_word,sex_cond,excerpt_src_fn,excerpt_cue_src_fn,excerpt_distractor_1_src_fn,excerpt_distractor_2_src_fn
0,601538,laura-s,0.29,1205.07,1204.78,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,1796.176689,each,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
1,638828,dolliellama,0.36,737.92,737.56,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,4871.649524,behavior,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
2,249863,ama1016,0.52,371.41,370.89,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,273.502041,language,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
3,127418,popularoutcast,0.47,3399.30,3398.83,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,1526.788934,ties,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...
4,961747,sedola,0.42,1898.92,1898.50,swc,female,0,NaN,0,...,0.0,44100.0,/scratch2/weka/mcdermott/msaddler/swc/english/...,2335.023311,from,same,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...,/om/user/imgriff/datasets/human_word_rec_SWC_2...


In [7]:
batch_size = 160 # 160 
n_iters = (len(src_fns) // batch_size) + 1 

transcripts = []

manifest = manifest.reset_index()
manifest.rename(columns={'index': 'manifest_ix'}, inplace=True)

print("Transcribing audio...")
for iter in tqdm(range(n_iters)):
    target_batch = src_fns[iter*batch_size : (iter+1)*batch_size]
    batch_ixs = np.arange(iter*batch_size, (iter+1)*batch_size, dtype='int')
    if len(batch_ixs) > len(target_batch):
        batch_ixs = batch_ixs[:len(target_batch)]
    # decode the audio
    target_audio = np.array([whisper.pad_or_trim(whisper.load_audio(fname)) for fname in target_batch])
    target_mel_specs = whisper.log_mel_spectrogram(target_audio).to(model.device)
    target_results = whisper.decode(model, target_mel_specs, options)

    # distrctor batch 
    # get first distractor results
    distractor_1_batch = distractor_1_fns[iter*batch_size : (iter+1)*batch_size]
    distractor_audio = np.array([whisper.pad_or_trim(whisper.load_audio(fname)) for fname in distractor_1_batch])
    distractor_mel_specs = whisper.log_mel_spectrogram(distractor_audio).to(model.device)
    distracted_1_results = whisper.decode(model, distractor_mel_specs, options)
    # get second distractor results
    distractor_2_batch = distractor_2_fns[iter*batch_size : (iter+1)*batch_size]
    distractor_audio = np.array([whisper.pad_or_trim(whisper.load_audio(fname)) for fname in distractor_2_batch])
    distractor_mel_specs = whisper.log_mel_spectrogram(distractor_audio).to(model.device)
    distracted_2_results = whisper.decode(model, distractor_mel_specs, options)

    # process text to lower case, remove punctuation, and convert digits to words
    batch_dict = {'src_fn': target_batch, 
            'distractor_1_src_fn': distractor_1_batch,
            'distractor_2_src_fn': distractor_2_batch,
            'manifest_ix': batch_ixs, 
            'target_transcripts': [convert_transcript(result.text) for result in target_results],
            'distractor_1_transcripts': [convert_transcript(result.text) for result in distracted_1_results],
            'distractor_2_transcripts': [convert_transcript(result.text) for result in distracted_2_results]}

    transcripts.append(batch_dict)

all_transcripts_df = pd.concat([pd.DataFrame.from_dict(batch_dict) for batch_dict in transcripts], axis=0, ignore_index=True)
ground_truth_df = manifest.merge(all_transcripts_df, left_on=['excerpt_src_fn', 'manifest_ix'], right_on=['src_fn', 'manifest_ix'], how='inner')


Transcribing audio...


100%|██████████| 13/13 [13:46<00:00, 63.61s/it]


ValueError: All arrays must be of the same length

In [ ]:
manifest.shape

(1952, 46)

In [27]:
assert (ground_truth_df.excerpt_src_fn == ground_truth_df.src_fn_y).all()
assert (ground_truth_df.index == ground_truth_df.manifest_ix).all()

In [28]:
## make sure target word is in transcript 
ground_truth_df[~ground_truth_df.apply(lambda x: x.word in ' '.join(x.target_transcripts), axis=1)][['word', 'target_transcripts']]

,word,target_transcripts
81,color,"[bu, bir, ok, eyin, bir]"
109,doctor,"[dr, giles, fletcher]"
135,exist,"[of, the, world]"
153,former,[rights]
263,moved,"[ill, move, back, with, the, allen]"
...,...,...
1906,twenty,"[four, and, on, the, 27th, of, june]"
1908,under,"[economy, ander, clinton, the, united, states,..."
1926,water,"[cwa, he, described]"
1927,website,"[it, was, webcast, on, the, bbc, iweb]"


In [30]:
### Save out transcripts:

ground_truth_df.to_pickle(out_manifest_name)


In [ ]:
out_manifest_name

PosixPath('/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl')